# 03 — Choix des dates NISAR (tâche D3)

Entrée : `inventaire_nisar_douala.csv` produit par le notebook D2 (à importer dans le panneau Fichiers de Colab).
Sortie : `dates_retenues.csv` — les dates à télécharger en D5.

**Règles de choix** (contrat de données de l'équipe) :
1. même **piste** + même **sens d'orbite** + même **niveau de maturité** + même produit ;
2. au moins une date en **saison sèche** (déc.–févr.) et une en **saison des pluies** (mars–nov.) ;
3. **couverture de la zone** élevée ;
4. on évite le trou de données du 27/07/2026 au 10/08/2026.

In [ ]:
import pandas as pd

CSV = "inventaire_nisar_douala.csv"
COUVERTURE_MIN = 90          # % de la zone couverte par la scène
TROU = ("2026-07-27", "2026-08-10")

df = pd.read_csv(CSV, dtype={"piste": str, "frame": str})
df = df[df["date"].notna()].copy()
print(len(df), "acquisitions lues")

## 1. Saison et maturité (déduites de la date)

La maturité est une **approximation par la date**. Vérifiez-la avec le champ réel de version/CRID affiché en D2.

In [ ]:
def saison(d):
    m = int(d[5:7])
    return "seche" if m in (12, 1, 2) else "pluies"

def maturite(d):
    if d < "2026-02-01":  return "BETA"
    if d < "2026-10-21":  return "PROVISIONAL"
    return "VALIDATED"

df["saison"] = df["date"].map(saison)
df["maturite"] = df["date"].map(maturite)
df["dans_trou"] = df["date"].between(*TROU)
print(df.groupby(["maturite", "saison"]).size())

## 2. Filtrage

In [ ]:
ok = df[(df["couverture_zone_%"] >= COUVERTURE_MIN) & (~df["dans_trou"])].copy()
print(f"{len(ok)} acquisitions retenues sur {len(df)} (couverture >= {COUVERTURE_MIN} %, hors trou de données)")

## 3. Groupes comparables, classés

In [ ]:
cles = ["piste", "orbite_dir", "produit", "maturite"]
lignes = []
for k, g in ok.groupby(cles, dropna=False):
    lignes.append({**dict(zip(cles, k)),
                   "n_dates": g["date"].nunique(),
                   "n_seche": g.loc[g.saison == "seche", "date"].nunique(),
                   "n_pluies": g.loc[g.saison == "pluies", "date"].nunique()})
groupes = pd.DataFrame(lignes)
if groupes.empty:
    print("Aucun groupe : voir le plan B (Sentinel-1, D6).")
else:
    groupes["utilisable"] = (groupes.n_seche > 0) & (groupes.n_pluies > 0)
    groupes = groupes.sort_values(["utilisable", "n_dates"], ascending=False).reset_index(drop=True)
groupes

## 4. Sélection proposée

Prend le meilleur groupe utilisable, puis pour chaque saison la date la mieux couverte.

In [ ]:
if groupes.empty or not groupes.utilisable.any():
    print("Aucun groupe avec saison sèche ET saison des pluies.")
    print("Options : accepter une comparaison pluies/pluies, élargir à une autre piste, ou plan B Sentinel-1 (D6).")
else:
    best = groupes[groupes.utilisable].iloc[0]
    g = ok.copy()
    for c in cles:
        g = g[g[c] == best[c]]
    choix = (g.sort_values("couverture_zone_%", ascending=False)
               .groupby(["saison", "date"], as_index=False).first()
               .sort_values(["saison", "couverture_zone_%"], ascending=[True, False])
               .groupby("saison").head(3).sort_values("date"))
    print("Groupe retenu :", dict(best[cles]))
    choix[["date", "saison", "piste", "orbite_dir", "produit", "maturite", "couverture_zone_%", "scene"]]

## 5. Sauvegarde

In [ ]:
if "choix" in globals():
    choix.to_csv("dates_retenues.csv", index=False)
    print("Enregistré : dates_retenues.csv")

## Fini quand (D3)
- [ ] Au moins 1 date sèche et 1 date des pluies, même piste / sens / maturité
- [ ] `dates_retenues.csv` partagé avec l'équipe
- [ ] La maturité a été vérifiée avec le champ de version réel du CSV